# Appendix A — Reproducible subsampling validation

Compare the canonical batch-4 subsample with its disjoint complement using
the final classifier and the same feature representation. Old numbers are not
treated as a new validation without a verified execution manifest.


In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'config.py').exists() and (p/'src').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import config
from src.features.cache import load_cache, write_cache
BATCH_VAL = 4
F_FULL = config.DIR_PROCESSED / f'features_e2gp_test_b{BATCH_VAL}_full.csv'


## 1. Full-batch feature cache

Validate an existing cache's representation, extractor, schema and content.
Otherwise compute the full batch without a cap. GP extraction is expensive;
an invalid cache causes an explicit error rather than silent reuse.


In [ ]:
if F_FULL.exists():
    full = load_cache(F_FULL, 'e2gp')
    print('Validated cache:', full.shape)
else:
    from src.pipeline import clean_batch
    from src.features import statistics as est, gp, gp2d
    batch_paths = config.BatchConfig(BATCH_VAL)
    curvas, metadata_table = clean_batch(batch_paths.test_lc, config.TEST_METADATA_CSV, max_obj=None)
    full = est.add_derived_features(
        est.build_feature_table(curvas, metadata_table)
        .merge(gp.build_gp_table(curvas, progress_interval=2000), on='object_id', validate='one_to_one')
        .merge(gp2d.build_gp2d_table(curvas, progress_interval=2000), on='object_id', validate='one_to_one'))
    write_cache(full, F_FULL, 'e2gp', producer_name='notebooks/appendices/A_subsampling_validation.ipynb')


## 2. Regenerate diagnostics and record provenance

Load the compatible final model, align shared rows to the canonical subsample
features in memory, and check that the model reproduces its saved probabilities.
Record medians, exploratory tests, Holm adjustment and 200 samples without
replacement. Write input, code and output hashes with software versions.


In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT/'scripts/regenerate_subsampling_validation.py'),
                '--batch', str(BATCH_VAL)], cwd=ROOT, check=True)


## 3. Interpret the regenerated tables

Examine effect sizes, adjusted tests, the central Monte Carlo intervals and
the position of the actual subsample. Failure to reject a difference is not
proof of identity. The empirical conclusion applies to batch 4 only.

The reference Linux execution on 23 October 2026 passed all 37 tests and regenerated these diagnostics. Median confidence is 0.743237 in the subsample and 0.746207 in its complement. The Holm-adjusted confidence p-value is 0.556821, and none of the seven nonredundant tests rejects at 5%. The subsample novelty ROC-AUC is 0.668225 (full batch: 0.693729), at percentile 7.5 of the 200 Monte Carlo draws. These checks do not establish statistical equivalence. Full provenance and the retained result archive are in reports/reproducibility/validation_20261023.md.


In [ ]:
from src.models.subsampling import load_validation
if BATCH_VAL != 4:
    raise ValueError('The main summary concerns batch 4; inspect batch-specific CSVs for another batch')
tests, summary_table, sample_records, procedencia = load_validation(config.DIR_PROCESSED)
display(tests.round(4))
display(summary_table.round(4))
print('Summary continuo:', procedencia['continuous_summary'])
print('Filas of features alineadas:', procedencia['aligned_feature_rows'])
